In [1]:
import pymaid
import navis
import pandas as pd
import numpy as np
import zarr
import docker
import tensorstore as ts
import os
import json
import boto3
from joblib import dump, load, Parallel, delayed
from ac_segmentation.neurotorch.datasets.dataset import open_ZarrTensor

In [ ]:
def extract_neuronlist(morph_in, node_min):
    def ind_neuron(tree):
        q = morph_in.nodes.query('node_id in @tree')
        neuron = navis.NeuronList(pd.DataFrame(q))
        return neuron
    
    trees = morph_in.subtrees
    res = Parallel(n_jobs=4)(delayed(ind_neuron)(tree) for tree in trees if len(tree) >= node_min)
    neurons = navis.NeuronList(res)
    for neu in neurons:
        neu.name = neu.id
        
    return neurons

In [ ]:
###import neurons into CATMAID from SWC file
def import_neurons(neurons, server, api_token, project_id, rad_annot=True, res=[1,1,1]):
    #Load instance and set project
    rm = pymaid.CatmaidInstance(server=server, api_token=api_token)
    rm.project_id = project_id

    #Import neurons
    for ns in neurons:
        nodes = pd.DataFrame(ns.nodes)
        nodes = nodes[['node_id', 'parent_id', 'x', 'y', 'z', 'radius']]
        neu = navis.TreeNeuron(nodes)
        if res != [1,1,1]:
            x,y,z = res
            neu.nodes['x'] = neu.nodes['x']*x
            neu.nodes['y'] = neu.nodes['y']*y
            neu.nodes['z'] = neu.nodes['z']*z
        resp = pymaid.upload_neuron(neu)
        if rad_annot==True:
            rad = round(max(list(neu.nodes['radius'])))
            resp = pymaid.add_annotations(int(resp['skeleton_id']), "RADIUS:" + str(rad))

In [ ]:
ns = navis.read_swc("/ACdata/Users/connorl/Skeletons/For_Kevin/S32_Pos52,53,54_MIP0/Skeletons/Pos52_Skels.swc")
ns.nodes = ns.nodes.rename(columns={"x": "z", "z": "x"})
ns = extract_neuronlist(ns, 40)

#import_neurons(neurons=ns, server="http://bigkahuna:4554/", api_token='7deede658f4bc1f53ee6993fc28ec26fbbe5838f', project_id = 20, res=[.705,.812,.812])

In [ ]:
###convert zarr to n5
def zarr_to_n5(zarr_path, out_path, out_name = "Data_Out", chunks=(64,64,64), cutout=None):
    #open zarr
    arr = open_ZarrTensor(zarr_path)
    if cutout != None:
        x1,x2,y1,y2,z1,z2 = cutout
        arr = arr[0,0,x1:x2,y1:y2,z1:z2].transpose().read().result()
    else:
        arr = arr[0,0,:,:,:].transpose().read().result()

    #create n5
    store = zarr.N5Store(os.path.join(out_path, out_name + '.n5'))
    root = zarr.group(store=store)
    z = root.zeros('group/' + zarr_path[-2], shape=arr.shape, chunks=chunks, dtype=arr.dtype, compressor=None)
    z[:] = arr

In [ ]:
#zarr_to_n5(zarr_path='/ACdata/Users/kevin/ispim_ome_zarr/H17_x55_S39a_230808_highres/H17_x55_S39a_230808_highres.zarr/highres_Pos90/1/', out_path='/ACdata/Users/connorl/N5_Files/', chunks=(64,64,64), cutout=[12000,16000,0,288,0,288])

In [ ]:
###convert zarr to precomputed
def zarr_to_precomputed_local(zarr_path, out_path, out_name = "Data_Out", chunks=(64,64,64), cutout=None, scales=6):
    #iterate over all scale levels
    for scale in range(0,scales):
        #open zarr
        arr = open_ZarrTensor(zarr_path+str(scale))
        if cutout!=None:
            x1,x2,y1,y2,z1,z2 = cutout
            arr = arr[0,0,x1:x2,y1:y2,z1:z2].read().result()
            cutout = list((np.array(cutout)/2).astype(int))
        else:
            arr = arr[0,0,:,:,:].read().result()
        arr = np.expand_dims(arr, axis=3)
    
        #get resolution
        r_path = os.path.join(os.path.dirname(zarr_path), ".zattrs")
        res = json.loads(open(r_path, "r").read())['multiscales'][0]['datasets'][int(scale)]['coordinateTransformations'][0]['scale'][2:]
            
        #create precomputed tensor
        pre_comp = ts.open(
                {
                    "driver": "neuroglancer_precomputed",
                    "kvstore": {"driver": "file","path": os.path.join(out_path, out_name)},
                    "scale_metadata": {
                        "resolution": res,
                        "chunk_size": list(chunks),
                        "encoding": "raw",
                        "key": "s" + str(scale)
                    }
                },
                create=True,
                dtype=arr.dtype,
                domain=ts.IndexDomain(
                    shape=list(arr.shape),
                )).result()
    
        pre_comp.write(arr).result()

def zarr_to_precomputed_S3(zarr_path, out_path, AWS_Key, AWS_Secret_Key, out_name = "Data_Out", chunks=(64,64,64), cutout=None, scales=6):
    #iterate over all scale levels
    for scale in range(0,scales):
        #open zarr
        arr = open_ZarrTensor(zarr_path+str(scale))
        if cutout!=None:
            x1,x2,y1,y2,z1,z2 = cutout
            arr = arr[0,0,x1:x2,y1:y2,z1:z2].read().result()
            cutout = list((np.array(cutout)/2).astype(int))
        else:
            arr = arr[0,0,:,:,:].read().result()
        arr = np.expand_dims(arr, axis=3)
    
        #get resolution
        r_path = os.path.join(os.path.dirname(zarr_path), ".zattrs")
        res = json.loads(open(r_path, "r").read())['multiscales'][0]['datasets'][int(scale)]['coordinateTransformations'][0]['scale'][2:]
            
        #create precomputed tensor
        bucket = out_path.split("/")[0]
        path = out_path.replace(bucket+"/", '') + out_name
        os.environ['AWS_ACCESS_KEY_ID']=AWS_Key
        os.environ['AWS_SECRET_ACCESS_KEY']=AWS_Secret_Key
        pre_comp = ts.open(
                {
                    "driver": "neuroglancer_precomputed",
                    "kvstore": {"driver": "s3","bucket": bucket ,"path": path},
                    "scale_metadata": {
                        "resolution": res,
                        "chunk_size": list(chunks),
                        "encoding": "raw",
                        "key": "s" + str(scale)
                    }
                },
                create=True,
                dtype=arr.dtype,
                domain=ts.IndexDomain(
                    shape=list(arr.shape),
                )).result()
    
        pre_comp.write(arr).result()

In [ ]:
AWS_Key=""
AWS_Secret_Key=""
#zarr_to_precomputed_S3(AWS_Key=AWS_Key, AWS_Secret_Key=AWS_Secret_Key, zarr_path='/ACdata/Users/kevin/ispim_ome_zarr/H17_x55_S32_230412_highres/H17_x55_S32_230412_highres.zarr/highres_Pos54/', out_path='ac-catmaid-dev/Data/S39a_Pos52_53_54/Image_Data/', out_name='testit', chunks=(64,64,64), cutout=[23000,27000,0,576,0,576])

In [ ]:
###create a precomputed volume for a given zarr in the designated S3 bucket, and import project data to CATMAID
def zarr_to_CATMAID_project(zarr_path, out_path, container_id, AWS_Key, AWS_Secret_Key, out_name='Data_Out', 
                                project='NewProject', stack='NewStack', chunks=(64,64,64), translation=(0,0,0), cutout=None, tile_dim=[128,128]):

    #convert zarr to precomputed
    zarr_to_precomputed_S3(zarr_path=zarr_path, AWS_Key=AWS_Key, AWS_Secret_Key=AWS_Secret_Key, 
                           out_path=out_path, out_name=out_name, chunks=(64,64,64), cutout=cutout)
    type_path = out_name + "/%SCALE_DATASET%/"
    
    #extract url 
    split = out_path.split("/")
    bucket = split[0]
    key = out_path.replace(bucket+"/", '')
    url = os.path.join("https://",bucket+".s3-us-west-2.amazonaws.com", key, type_path)
    
    #get resolution and shape
    shape = open_ZarrTensor(zarr_path + "0/")[0,0,:,:,:].shape
    if cutout!=None:
        x1,x2,y1,y2,z1,z2 = cutout
        x,y,z = x2-x1,y2-y1,z2-z1
        shape = [x,y,z]
    r_path = os.path.join(os.path.dirname(os.path.dirname(zarr_path + "0/")), ".zattrs")
    res = json.loads(open(r_path, "r").read())['multiscales'][0]['datasets'][0]['coordinateTransformations'][0]['scale'][2:]
    
    #create project data json
    stack_file = [{
      "project": {
        "title": project,
        "stacks": [{
          "title": stack+"xy",
          "dimension": str(tuple(shape)),
          "mirrors": [{
            "fileextension": "raw",
            "position": 0,
            "tile_source_type": 14,
            "tile_height":tile_dim[0],
            "tile_width":tile_dim[1],
            "title": "Example tiles",
            "url": url + "0_1_2"
          }],
          "resolution": str(tuple(res)),
          "translation": str(translation),
          "downsample_factors": ["(1,1,1)", "(2,2,2)", "(4,4,4)"],
          "orientation" : 0,
          "stackgroups": [{"title": "Project5_StackGroup", "relation": "channel"}]
        },
        {
          "title": stack+"xz",
          "dimension": str(tuple([shape[0],shape[2],shape[1]])),
          "mirrors": [{
            "fileextension": "raw",
            "position": 0,
            "tile_source_type": 14,
            "tile_height":tile_dim[0],
            "tile_width":tile_dim[1],
            "title": "Example tiles",
            "url": url + "0_2_1"
          }],
          "resolution": str(tuple([res[0],res[2],res[1]])),
          "translation": str(tuple([translation[0],translation[2],translation[1]])),
          "downsample_factors": ["(1,1,1)", "(2,2,2)", "(4,4,4)"],
          "orientation" : 1,
          "stackgroups": [{"title": "Project5_StackGroup", "relation": "view"}]
        },
        {
          "title": stack+"zy",
          "dimension": str(tuple([shape[2],shape[1],shape[0]])),
          "mirrors": [{
            "fileextension": "raw",
            "position": 0,
            "tile_source_type": 14,
            "tile_height":tile_dim[0],
            "tile_width":tile_dim[1],
            "title": "Example tiles",
            "url": url + "2_1_0"
          }],
          "resolution": str(tuple([res[2],res[1],res[0]])),
          "translation": str(tuple([translation[2],translation[1],translation[0]])),
          "downsample_factors": ["(1,1,1)", "(2,2,2)", "(4,4,4)"],
          "orientation" : 2,
          "stackgroups": [{"title": "Project5_StackGroup", "relation": "view"}]
        }]
      }
    }]
    
    #save json to local
    json_fpath = os.path.join("./", "CATMAID_Project.json")
    with open(json_fpath, 'w') as f:
        json.dump(stack_file, f)

    #copy json file to container
    copy_string = json_fpath + " " + container_id + ":" + "/home/django/projects/data.json"
    ! docker cp $copy_string
    
    #open docker container and impo9rt json
    client = docker.from_env()
    container = client.containers.get(container_id)
    container.exec_run('python3 manage.py catmaid_import_projects --input data.json --permission user:admin:can_import user:admin:can_annotate')

    #clean-up
    container.exec_run('rm input CATMAID_Project.json')
    client.close()
    os.remove(json_fpath)

    return stack_file

In [ ]:
AWS_Key=""
AWS_Secret_Key=""
#stack = zarr_to_CATMAID_project(AWS_Key=AWS_Key, AWS_Secret_Key=AWS_Secret_Key,zarr_path="/ACdata/Users/kevin/ispim_ome_zarr/H17_x55_S39a_230808_highres/H17_x55_S39a_230808_highres.zarr/highres_Pos56/", out_path='ac-catmaid-dev/Data/S39a_Pos52_53_54/Image_Data/', container_id='9024a7f6d341', cutout=[23000,27000,0,576,0,576], project='TestProject', stack='TestStack')